In [4]:
from datetime import datetime, time
from Database.TPData import TPData, TPDataDa
from OrderBook.OrderBook import OrderBookSnaps
from SynthSpread.spreadviewer_class import SpreadSingle
from Strategies.Sparse_momentum.ob_attributes import OB_attributes, TR_attributes
import pandas as pd
import numpy as np
from Utilities.excel_loaders import conn_out_xload_mac
from Utilities.dfutils import dict_iloc
from Utilities.func_utils import load_arguments
import pickle

def variables_from_instrument(instrument: str):
    result = {
        'mkt': None,
        'tenor': None,
        'tn': None
    }
    for x in ['de', 'fr', 'ttf']:
        if x in instrument:
            result['mkt'] = x
    result['tenor'] = instrument[-2]
    result['tn'] = int(instrument[-1])
    return result


if __name__ == '__main__':
    dates_out = conn_out_xload_mac()
    allwd_broker_ids = [1441]

    # ------------------ dataset prep ---------------------------------
    # Load arguments
    _INSTRUMENTS = ['dem1']
    _START_DATE, _END_DATE = '2025-03-01', '2025-03-05'

    # setting variables
    ins_dicts = [variables_from_instrument(x) for x in _INSTRUMENTS]
    n_s = 2
    mkt_list = [ins_dict['mkt'] for ins_dict in ins_dicts]
    tenor_list = [ins_dict['tenor'] for ins_dict in ins_dicts]
    tn1_list = [ins_dict['tn'] for ins_dict in ins_dicts]
    ts_lag = (lambda i: mkt_list[i] + tenor_list[i] + str(tn1_list[i]))(0)
    
    tn2_list = []
    prod = 'base'
    venue_list = ['eex']
    start_date = datetime.strptime(_START_DATE, '%Y-%m-%d').date()
    end_date = datetime.strptime(_END_DATE, '%Y-%m-%d').date()
    # ---------------------------------------------------------------------
    
    if not tn2_list:
        tn_list = [str(t1) for t1 in tn1_list]
    else:
        tn_list = [str(t1) + '_' + str(t2) for (t1, t2) in zip(tn1_list, tn2_list)]
    
    dates = pd.date_range(start_date, end_date, freq='B')
    
    spread_class = SpreadSingle(mkt_list, tenor_list, tn1_list, tn2_list, venue_list)
    product_date1 = spread_class.product_dates(dates, n_s, tn_bool=True)
    product_date2 = spread_class.product_dates(dates, n_s, tn_bool=False)
    
    start_time = time(9, 0, 0, 0)
    end_time = time(17, 40, 0, 0)
    
    gran = None
    
    
    data_class = TPData()

    trades_dict = {}

    for k, ds in enumerate(dates):
        if ds in dates_out:
            continue
        bT = datetime.combine(ds, start_time)
        eT = datetime.combine(ds, end_time)
        pd1_aux = [None if p is None else p[k] for p in product_date1]
        pd2_aux = [None if p is None else p[k] for p in product_date2]
        for (m, t, n, pd1, pd2) in zip(mkt_list, tenor_list, tn_list,
                                                    pd1_aux, pd2_aux):
            i = m + t + str(n)
            # Trades
            data_class.create_connection('OracleSQL')
            trades = data_class.get_trades(m, t, venue_list, pd1, bT, eT,
                                                prod)
            trades = trades[trades['broker_id'].isin(allwd_broker_ids)]
            trades = trades.reset_index(names='datetime').drop_duplicates('datetime', keep='last').set_index('datetime')

            if i not in trades_dict:
                trades_dict[i] = trades
            else:
                trades_dict[i] = pd.concat([trades_dict[i], trades])



Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle


In [5]:
trades_dict['dem1']

,tradeid,price,volume,action,broker_id,own_trades
datetime,,,,,,
2025-03-03 09:00:00.018847829,Eurex T7/DEBM042025-20250303/881/1,81.96,1,1,1441,False
2025-03-03 09:00:19.931431884,Eurex T7/DEBM042025-20250303/883/1,81.70,1,-1,1441,False
2025-03-03 09:00:23.164436074,Eurex T7/DEBM042025-20250303/885/2,81.70,2,-1,1441,False
2025-03-03 09:00:26.187842672,Eurex T7/DEBM042025-20250303/887/1,81.71,1,-1,1441,False
2025-03-03 09:00:42.474779837,Eurex T7/DEBM042025-20250303/896/1,81.83,1,1,1441,False
...,...,...,...,...,...,...
2025-03-05 17:38:04.364518912,Eurex T7/DEBM042025-20250305/12656/1,75.25,1,-1,1441,False
2025-03-05 17:38:04.365804366,Eurex T7/DEBM042025-20250305/12657/1,75.24,1,1,1441,False
2025-03-05 17:38:16.592589199,Eurex T7/DEBM042025-20250305/12663/1,75.25,1,-1,1441,False
